# Assign each complaint a dominant topic

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

file_path = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_text_clean.csv"
output_file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_topics.csv"

df = pd.read_csv(file_path, low_memory=False)

vectorizer = TfidfVectorizer(
    max_features=2000,
    min_df=10,
    max_df=0.90,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean_text"])

n_topics = 8
nmf_model = NMF(n_components=n_topics, random_state=42)
topic_matrix = nmf_model.fit_transform(X)

# Dominant topic index
df["topic_number"] = topic_matrix.argmax(axis=1) + 1

# Manual business labels — update after inspecting topic keywords
topic_labels = {
    1: "Credit Reporting Errors",
    2: "Consumer Rights Violations",
    3: "Fintech Payment Platform Issues",
    4: "Bank Account & Card Transaction Issues",
    5: "Debt Collection Disputes",
    6: "Consumer Reporting Agency Issues",
    7: "Digital Transfer Fraud (Zelle)",
    8: "Identity Theft & Fraud"
}

df["topic_label"] = df["topic_number"].map(topic_labels)

df.to_csv(output_file, index=False)

print("Saved topic-assigned complaints to:")
print(output_file)

print(df[["complaint_id", "issue", "topic_number", "topic_label"]].head())

Saved topic-assigned complaints to:
C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_topics.csv
   complaint_id                                              issue  \
0       4286092                          Other transaction problem   
1       6116458  Problem with a credit reporting company's inve...   
2       7405732                  Attempts to collect debt not owed   
3      10490489                     Trouble during payment process   
4      11397302                  Attempts to collect debt not owed   

   topic_number                             topic_label  
0             4  Bank Account & Card Transaction Issues  
1             4  Bank Account & Card Transaction Issues  
2             4  Bank Account & Card Transaction Issues  
3             4  Bank Account & Card Transaction Issues  
4             4  Bank Account & Card Transaction Issues  


In [3]:
topic_summary = (
    df.groupby("topic_label")
      .size()
      .reset_index(name="complaint_count")
      .sort_values("complaint_count", ascending=False)
)

print(topic_summary)

                              topic_label  complaint_count
0  Bank Account & Card Transaction Issues             9714
3                 Credit Reporting Errors             6032
4                Debt Collection Disputes             3867
7                  Identity Theft & Fraud             3501
5          Digital Transfer Fraud (Zelle)             2984
2              Consumer Rights Violations             2551
6         Fintech Payment Platform Issues              823
1        Consumer Reporting Agency Issues              525


In [4]:
df.to_csv(
r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_with_topics.csv",
index=False
)